import stuff

In [1]:
import sys
# Add the local stonesoup directory to sys.path
project_path = r"C:\Users\joesb\Documents\stonesoup"  # Adjust this to your actual path
if project_path not in sys.path:
    sys.path.insert(0, project_path)

# print(sys.path)

import numpy as np
from datetime import datetime, timedelta

from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel


simulate groundtruth

In [2]:
# And the clock starts
start_time = datetime.now().replace(microsecond=0)

seed = 1 # Random seem for reproducibility

# Driving process parameters
mu_W = 0
sigma_W2 = 4
alpha = 1.4
c=10

# Model parameters
theta=0.15

driver_x = AlphaStableNSMDriver(mu_W=mu_W, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha)

driver_y = driver_x # Same driving process in both dimensions and sharing the same latents (jumps)
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta, mu_W=-0.02)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])

from ordered_set import OrderedSet
np.random.seed(1991)

truths = OrderedSet()

num_steps = 50
timesteps = [start_time]
truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])
for k in range(num_steps):
    timesteps.append(start_time+timedelta(seconds=1*(k+1)))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k+1]))
truths.add(truth)

remaining_start_states = [[0, 1, 5000, -1],[5000, 1, 0, -1],[5000, 1, 5000, -1],[-5000, 1, -5000, -1]]
for state in remaining_start_states:
    truth = GroundTruthPath([GroundTruthState(state, timestamp=timesteps[0])])
    for k in range(num_steps):
        truth.append(GroundTruthState(
            transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1)),
            timestamp=timesteps[k+1]))
    truths.add(truth)


generate detections

In [3]:
from scipy.stats import uniform

from stonesoup.types.detection import TrueDetection
from stonesoup.types.detection import Clutter
from stonesoup.models.measurement.linear import LinearGaussian

x_noise=5
y_noise=x_noise
measurement_model = LinearGaussian(
    ndim_state=4,
    mapping=(0, 2),
    noise_covar=np.array([[x_noise, 0],
                          [0, y_noise]])
    )
all_measurements = []

for k in range(num_steps):
    measurement_set = set()

    for truth in truths:
        # Generate actual detection from the state with a 10% chance that no detection is received.
        if np.random.rand() <= 0.9:
            measurement = measurement_model.function(truth[k], noise=True)
            measurement_set.add(TrueDetection(state_vector=measurement,
                                              groundtruth_path=truth,
                                              timestamp=truth[k].timestamp,
                                              measurement_model=measurement_model))

        # Generate clutter at this time-step
        truth_x = truth[k].state_vector[0]
        truth_y = truth[k].state_vector[2]
        for _ in range(np.random.randint(4)):
            x = uniform.rvs(truth_x - x_noise, 2*x_noise)
            y = uniform.rvs(truth_y - y_noise, 2*y_noise)
            measurement_set.add(Clutter(np.array([[x], [y]]), timestamp=truth[k].timestamp,
                                        measurement_model=measurement_model))
    all_measurements.append(measurement_set)

from stonesoup.plotter import AnimatedPlotterly
plotter = AnimatedPlotterly(timesteps, tail_length=1)
plotter.plot_ground_truths(truths, [0, 2])
# Plot true detections and clutter.
plotter.plot_measurements(all_measurements, [0, 2])
plotter.fig

generate 5 prior states

In [4]:
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import StateVectors

number_particles = 50

# Sample from the prior Gaussian distribution
states1 = multivariate_normal.rvs(np.array([0, 1, 0, 1]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
states2 = multivariate_normal.rvs(np.array(remaining_start_states[0]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
states3 = multivariate_normal.rvs(np.array(remaining_start_states[1]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
states4 = multivariate_normal.rvs(np.array(remaining_start_states[2]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
states5 = multivariate_normal.rvs(np.array(remaining_start_states[3]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)

covars = np.stack([np.eye(4) * 100 for i in range(number_particles)], axis=2) # (M, M, N)
# Create prior particle states.

prior1 = MarginalisedParticleState(
    state_vector=StateVectors(states1.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

prior2 = MarginalisedParticleState(
    state_vector=StateVectors(states2.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

prior3 = MarginalisedParticleState(
    state_vector=StateVectors(states3.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))
prior4 = MarginalisedParticleState(
    state_vector=StateVectors(states4.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))
prior5 = MarginalisedParticleState(
    state_vector=StateVectors(states5.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

generate tracking estimations

In [5]:
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler 
from stonesoup.updater.particle import MarginalisedParticleUpdater

predictor = MarginalisedParticlePredictor(transition_model=transition_model)
resampler = SystematicResampler()
updater = MarginalisedParticleUpdater(measurement_model, resampler)

from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.measures import Mahalanobis
from stonesoup.dataassociator.neighbour import GlobalNearestNeighbour, GNNWith2DAssignment
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

hypothesiser = DistanceHypothesiser(predictor, updater, measure=Mahalanobis())
data_associator = GlobalNearestNeighbour(hypothesiser)
# data_associator = GNNWith2DAssignment(hypothesiser)

track1=Track()
track1.append(prior1)
track2=Track()
track2.append(prior2)
track3=Track()
track3.append(prior3)
track4=Track()
track4.append(prior4)
track5=Track()
track5.append(prior5)
tracks = {track1,track2,track3,track4,track5}

for n, measurements in enumerate(all_measurements):
    # Calculate all hypothesis pairs and associate the elements in the best subset to the tracks.
    hypotheses = data_associator.associate(tracks=tracks,
                                           detections=measurements,
                                           timestamp=start_time + timedelta(seconds=n))
    for j, track in enumerate(tracks):
        hypothesis = hypotheses[track]
        if hypothesis.measurement:
            post = updater.update(hypothesis)
            track.append(post)
        else:  # When data associator says no detections are good enough, we'll keep the prediction
            track.append(hypothesis.prediction)
            print(f"  Track {j+1}: No Detection Assigned - Keeping Prediction {hypothesis.prediction.state_vector.mean}")
    print(f"iteration{n+1} of {len(all_measurements)}")


iteration1 of 50
iteration2 of 50
iteration3 of 50
iteration4 of 50
iteration5 of 50
iteration6 of 50
iteration7 of 50
iteration8 of 50
iteration9 of 50
iteration10 of 50
iteration11 of 50
iteration12 of 50
iteration13 of 50
iteration14 of 50
iteration15 of 50
iteration16 of 50
iteration17 of 50
iteration18 of 50
iteration19 of 50
iteration20 of 50
iteration21 of 50
iteration22 of 50
iteration23 of 50
iteration24 of 50
iteration25 of 50
iteration26 of 50
iteration27 of 50
iteration28 of 50
iteration29 of 50
iteration30 of 50
iteration31 of 50
iteration32 of 50
iteration33 of 50
iteration34 of 50
iteration35 of 50
iteration36 of 50
iteration37 of 50
iteration38 of 50
iteration39 of 50
iteration40 of 50
iteration41 of 50
iteration42 of 50
iteration43 of 50
iteration44 of 50
iteration45 of 50
iteration46 of 50
iteration47 of 50
iteration48 of 50
iteration49 of 50
iteration50 of 50


plot it all

In [6]:
from stonesoup.plotter import AnimatedPlotterly, Plotterly, Dimension
from pathlib import Path

plotter = AnimatedPlotterly(timesteps, tail_length=1)
plotter.plot_ground_truths(truths, [0, 2])
# Plot true detections and clutter.
plotter.plot_measurements(all_measurements, [0, 2])
plotter.plot_tracks(tracks, [0, 2], uncertainty=True)
plotter.fig



In [7]:
axis_label_list=["x","dx_dt","y","dy_dt"]
particle_plotter_dict = {}

for i,label in enumerate(axis_label_list):
    particle_plotter_dict[label]= Plotterly(autosize=False, width=1200,height=600, dimension=Dimension.ONE, axis_labels=[label])
    # if label =="x" or label=="y":
        # particle_plotter_dict[label].plot_measurements(all_measurements, [i],marker=dict(symbol="x",size=5))
    for j,truth in enumerate(truths):
        particle_plotter_dict[label].plot_ground_truths(truth, [i],mode="lines", truths_label=f"truth {j+1}",line=dict(width=1))
    for j,track in enumerate(tracks):
        particle_plotter_dict[label].plot_tracks(track, [i],mode="lines", track_label=f"track {j+1}", line=dict(width=1))
    particle_plotter_dict[label].fig.show()
    file_path = Path(rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\TMR\multiobjectplot1D{label}.html")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    particle_plotter_dict[label].fig.write_html(str(file_path))

Test comparator from stonesoup website for gaussian tracker

In [8]:
from datetime import datetime, timedelta
start_time = datetime.now().replace(microsecond=0)

from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, \
                                               ConstantVelocity
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from ordered_set import OrderedSet

np.random.seed(1991)

truths = OrderedSet()

transition_model = CombinedLinearGaussianTransitionModel([ConstantVelocity(0.005),
                                                          ConstantVelocity(0.005)])

timesteps = [start_time]
truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])
for k in range(1, 21):
    timesteps.append(start_time+timedelta(seconds=k))
    truth.append(GroundTruthState(
        transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k]))
truths.add(truth)

truth = GroundTruthPath([GroundTruthState([0, 1, 20, -1], timestamp=timesteps[0])])
for k in range(1, 21):
    truth.append(GroundTruthState(
        transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k]))
_ = truths.add(truth)

from stonesoup.plotter import AnimatedPlotterly
plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_ground_truths(truths, [0, 2])

from scipy.stats import uniform

from stonesoup.types.detection import TrueDetection
from stonesoup.types.detection import Clutter
from stonesoup.models.measurement.linear import LinearGaussian

measurement_model = LinearGaussian(
    ndim_state=4,
    mapping=(0, 2),
    noise_covar=np.array([[0.75, 0],
                          [0, 0.75]])
    )
all_measurements = []

for k in range(20):
    measurement_set = set()

    for truth in truths:
        # Generate actual detection from the state with a 10% chance that no detection is received.
        if np.random.rand() <= 0.9:
            measurement = measurement_model.function(truth[k], noise=True)
            measurement_set.add(TrueDetection(state_vector=measurement,
                                              groundtruth_path=truth,
                                              timestamp=truth[k].timestamp,
                                              measurement_model=measurement_model))

        # Generate clutter at this time-step
        truth_x = truth[k].state_vector[0]
        truth_y = truth[k].state_vector[2]
        for _ in range(np.random.randint(10)):
            x = uniform.rvs(truth_x - 10, 20)
            y = uniform.rvs(truth_y - 10, 20)
            measurement_set.add(Clutter(np.array([[x], [y]]), timestamp=truth[k].timestamp,
                                        measurement_model=measurement_model))
    all_measurements.append(measurement_set)

# Plot true detections and clutter.
plotter.plot_measurements(all_measurements, [0, 2])



from stonesoup.predictor.kalman import KalmanPredictor
predictor = KalmanPredictor(transition_model)

from stonesoup.updater.kalman import KalmanUpdater
updater = KalmanUpdater(measurement_model)

from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.measures import Mahalanobis
hypothesiser = DistanceHypothesiser(predictor, updater, measure=Mahalanobis(), missed_distance=3)


from stonesoup.dataassociator.neighbour import GlobalNearestNeighbour
data_associator = GlobalNearestNeighbour(hypothesiser)

from stonesoup.types.state import GaussianState
prior1 = GaussianState([[0], [1], [0], [1]], np.diag([1.5, 0.5, 1.5, 0.5]), timestamp=start_time)
prior2 = GaussianState([[0], [1], [20], [-1]], np.diag([1.5, 0.5, 1.5, 0.5]), timestamp=start_time)

from stonesoup.types.track import Track
tracks = {Track([prior1]), Track([prior2])}

for n, measurements in enumerate(all_measurements):
    # Calculate all hypothesis pairs and associate the elements in the best subset to the tracks.
    hypotheses = data_associator.associate(tracks,
                                           measurements,
                                           start_time + timedelta(seconds=n))
    for track in tracks:
        hypothesis = hypotheses[track]
        if hypothesis.measurement:
            post = updater.update(hypothesis)
            track.append(post)
        else:  # When data associator says no detections are good enough, we'll keep the prediction
            track.append(hypothesis.prediction)


plotter.plot_tracks(tracks, [0, 2], uncertainty=True)
plotter.fig
